# Bates Model — Deep Hedging and Adversarial Robustness

Extends the Heston experiments to the **Bates model** (Heston SV + compound Poisson jumps).

Four experiments:
1. **Bates Baseline**: Vanilla vs W-DRO training on Bates paths
2. **Cross-Model Robustness**: Heston-trained models evaluated on Bates paths
3. **Jump Intensity Stress Test**: CVaR vs λ for all four models
4. **Adversarial Attacks on Bates**: S-WPGD and SV-WPGD attack tables

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set QUICK_RUN=False for the full experiment. True runs a fast validation (~5 min).
QUICK_RUN = True

if QUICK_RUN:
    N_TR           = 5_000
    N_TE           = 1_000
    N_EPOCHS_CLEAN = 50
    N_EPOCHS_ADV   = 30
    N_STRESS_TE    = 2_000    # paths per lambda stress test
    ATK_N_TRAIN    = 3
    ATK_N_EVAL     = 5
else:
    N_TR           = 40_000
    N_TE           = 10_000
    N_EPOCHS_CLEAN = 300
    N_EPOCHS_ADV   = 200
    N_STRESS_TE    = 10_000
    ATK_N_TRAIN    = 5
    ATK_N_EVAL     = 20

BATCH_SIZE = 5_000
LR_CLEAN   = 5e-2
LR_ADV     = 5e-3
ALPHA_BAL  = 1.0       # α in L_adv = α·L_clean + L_adv_paths

# Attack hyperparameters
ATK_DELTA  = 0.1
ATK_RATIO  = 4.0
DELTA_GRID = [0.0, 0.01, 0.03, 0.05, 0.10, 0.30, 0.50]

# Option parameters
ALPHA_CVAR = 0.95
K          = 100.0
N_STEPS    = 30

# Bates calibrated parameters — Kozyra Table 4.1
BATES_S0      = 100.0
BATES_V0      = 0.0574
BATES_KAPPA   = 0.4963
BATES_THETA   = 0.0650
BATES_XI      = 0.2286
BATES_RHO     = -0.990
BATES_MU_J    = 0.1791
BATES_SIGMA_J = 0.1346
BATES_LAM     = 0.1382

# Heston calibrated parameters — Kozyra Table 4.1
HESTON_S0    = 100.0
HESTON_V0    = 0.0654
HESTON_KAPPA = 0.6067
HESTON_THETA = 0.0707
HESTON_XI    = 0.2928
HESTON_RHO   = -0.7571

# Lambda stress grid (Exp 3)
LAMBDA_GRID = [0.14, 0.20, 0.30, 0.50, 0.75, 1.00]

In [ ]:
# ── Imports & Device ───────────────────────────────────────────────────────────
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.insert(0, '..')
from src.bates_simulator import BatesParams, BatesSimulator
from src.heston_simulator import HestonParams, HestonSimulator
from src.hedging.bates_network import BatesHedgeNet
from src.hedging.bates_loss import BatesCVaRLoss
from src.hedging.bates_trainer import train as bates_train
from src.hedging.bates_attacker import BatesDHAttacker

# Only for loading the pre-trained Heston vanilla model (Exp 2)
from src.hedging.hedge_network import HestonHedgeNet

torch.set_float32_matmul_precision('high')

device = (
    torch.device('mps')  if torch.backends.mps.is_available()  else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
RESULTS_DIR = Path('..') / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Device     : {device}")
print(f"QUICK_RUN  : {QUICK_RUN}")
print(f"N_TR={N_TR:,}  N_TE={N_TE:,}  N_EPOCHS_CLEAN={N_EPOCHS_CLEAN}  N_EPOCHS_ADV={N_EPOCHS_ADV}")

In [ ]:
# ── Simulate Paths ─────────────────────────────────────────────────────────────
print("Simulating Bates training paths ...")
t0 = time.perf_counter()
bates_params_tr = BatesParams(
    S0=BATES_S0, v0=BATES_V0, kappa=BATES_KAPPA, theta=BATES_THETA,
    xi=BATES_XI, rho=BATES_RHO, mu_J=BATES_MU_J, sigma_J=BATES_SIGMA_J,
    lam=BATES_LAM, T=N_STEPS/365, N=N_STEPS, M=N_TR,
)
S_btr, V_btr, VP_btr_raw = BatesSimulator(bates_params_tr).simulate(seed=10)
print(f"  {N_TR:,} Bates training paths  ({time.perf_counter()-t0:.1f}s)")

# VarPrice scaling — CRITICAL: aligns gradient magnitudes for δ_S and δ_V.
bates_vp_scale = 1.0 / float(VP_btr_raw[:, 0].mean())
VP_btr = VP_btr_raw * bates_vp_scale
print(f"  bates_vp_scale = {bates_vp_scale:.2f}")

print("\nSimulating Bates test paths ...")
t0 = time.perf_counter()
bates_params_te = BatesParams(
    S0=BATES_S0, v0=BATES_V0, kappa=BATES_KAPPA, theta=BATES_THETA,
    xi=BATES_XI, rho=BATES_RHO, mu_J=BATES_MU_J, sigma_J=BATES_SIGMA_J,
    lam=BATES_LAM, T=N_STEPS/365, N=N_STEPS, M=N_TE,
)
S_bte, V_bte, VP_bte_raw = BatesSimulator(bates_params_te).simulate(seed=11)
VP_bte = VP_bte_raw * bates_vp_scale
print(f"  {N_TE:,} Bates test paths  ({time.perf_counter()-t0:.1f}s)")

print("\nSimulating Heston training paths ...")
t0 = time.perf_counter()
heston_params_tr = HestonParams(
    S0=HESTON_S0, v0=HESTON_V0, kappa=HESTON_KAPPA, theta=HESTON_THETA,
    xi=HESTON_XI, rho=HESTON_RHO, T=N_STEPS/365, N=N_STEPS, M=N_TR,
)
S_htr, V_htr, VP_htr_raw = HestonSimulator(heston_params_tr).simulate(seed=20)
heston_vp_scale = 1.0 / float(VP_htr_raw[:, 0].mean())
VP_htr = VP_htr_raw * heston_vp_scale
print(f"  {N_TR:,} Heston training paths  ({time.perf_counter()-t0:.1f}s)")
print(f"  heston_vp_scale = {heston_vp_scale:.2f}")

print("\nSimulating Heston test paths ...")
heston_params_te = HestonParams(
    S0=HESTON_S0, v0=HESTON_V0, kappa=HESTON_KAPPA, theta=HESTON_THETA,
    xi=HESTON_XI, rho=HESTON_RHO, T=N_STEPS/365, N=N_STEPS, M=N_TE,
)
S_hte, V_hte, VP_hte_raw = HestonSimulator(heston_params_te).simulate(seed=21)
VP_hte = VP_hte_raw * heston_vp_scale
print(f"  {N_TE:,} Heston test paths")

# p0 warm-starts
with torch.no_grad():
    bates_p0_init  = float(torch.clamp(S_btr[:, -1] - K, min=0.0).mean())
    heston_p0_init = float(torch.clamp(S_htr[:, -1] - K, min=0.0).mean())
print(f"\nBates  call price (MC) ≈ {bates_p0_init:.4f}")
print(f"Heston call price (MC) ≈ {heston_p0_init:.4f}")

In [ ]:
# ── Shared Helper Functions ────────────────────────────────────────────────────
loss_fn = BatesCVaRLoss(K=K, alpha=ALPHA_CVAR)


def make_input(S: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """(batch, N+1) × 2 → (batch, N, 2) = [log S_t, V_t] for t=0..N-1."""
    return torch.cat([
        torch.log(S[:, :-1]).unsqueeze(-1),
        V[:, :-1].unsqueeze(-1),
    ], dim=-1)


@torch.no_grad()
def compute_hedging_errors(
    net: nn.Module,
    S: torch.Tensor,
    V: torch.Tensor,
    VarPrice: torch.Tensor,
    batch_size: int = 5_000,
) -> torch.Tensor:
    """Returns (M,) CPU tensor of per-path hedging errors X = C_T − PnL."""
    net = net.to(device).eval()
    errors = []
    for i in range(0, S.shape[0], batch_size):
        S_b  = S[i:i+batch_size].to(device)
        V_b  = V[i:i+batch_size].to(device)
        VP_b = VarPrice[i:i+batch_size].to(device)
        h    = net(make_input(S_b, V_b))
        dS   = S_b[:, 1:] - S_b[:, :-1]
        dVP  = VP_b[:, 1:] - VP_b[:, :-1]
        PnL  = (h[:, :, 0] * dS + h[:, :, 1] * dVP).sum(1)
        X    = loss_fn.terminal_payoff(S_b[:, -1]) - PnL
        errors.append(X.cpu())
    return torch.cat(errors)


def empirical_ES(errors: torch.Tensor, alpha: float = ALPHA_CVAR) -> float:
    """ES_alpha: mean of the top (1−alpha) fraction of losses."""
    k = max(1, int(np.ceil((1.0 - alpha) * len(errors))))
    return float(torch.topk(errors, k).values.mean())


def eval_metrics(errors: torch.Tensor) -> dict:
    """Returns dict with mean, std, var95, var99, cvar95."""
    e = errors.numpy()
    return {
        "mean":   float(e.mean()),
        "std":    float(e.std()),
        "var95":  float(np.quantile(e, 0.95)),
        "var99":  float(np.quantile(e, 0.99)),
        "cvar95": empirical_ES(errors, alpha=0.95),
    }


def _lr_lambda(epoch: int) -> float:
    """LR multiplier — mirrors bates_trainer.py."""
    if epoch < 200: return 1.0
    if epoch < 400: return 0.1
    if epoch < 600: return 0.01
    return 0.001


print("Helper functions defined.")

In [ ]:
# ── Training Functions ─────────────────────────────────────────────────────────
attacker = BatesDHAttacker()


def train_vanilla(
    S: torch.Tensor,
    V: torch.Tensor,
    VP: torch.Tensor,
    p0_init: float,
    n_epochs: int,
    lr: float = LR_CLEAN,
    verbose: bool = True,
):
    """Train BatesHedgeNet on clean paths. Returns (network_cpu, p0_scalar)."""
    net = BatesHedgeNet(N=N_STEPS, width=20).to(device)
    p0  = nn.Parameter(torch.tensor(p0_init, dtype=torch.float32, device=device))
    opt = torch.optim.Adam(
        [{'params': net.parameters()}, {'params': [p0]}], lr=lr
    )
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=_lr_lambda)
    S_d, V_d, VP_d = S.to(device), V.to(device), VP.to(device)
    M = S_d.shape[0]

    for epoch in range(n_epochs):
        net.train()
        idx = torch.randperm(M, device=device)[:min(BATCH_SIZE, M)]
        h   = net(make_input(S_d[idx], V_d[idx]))
        l   = loss_fn(h, S_d[idx], VP_d[idx], p0)
        opt.zero_grad(); l.backward(); opt.step(); sched.step()
        if verbose and (epoch % max(n_epochs // 5, 1) == 0 or epoch == n_epochs - 1):
            print(f"  epoch {epoch:4d}  loss={l.item():.4f}  p0={p0.item():.4f}")

    return net.cpu(), float(p0.item())


def train_wdro(
    S: torch.Tensor,
    V: torch.Tensor,
    VP: torch.Tensor,
    p0_init: float,
    verbose: bool = True,
):
    """
    Two-stage W-DRO adversarial training.

    Stage 1 — clean pre-training (N_EPOCHS_CLEAN epochs).
    Stage 2 — adversarial fine-tuning (N_EPOCHS_ADV epochs):
        L_total = ALPHA_BAL * L_clean + L_adv_paths

    Returns (network_cpu, p0_scalar).
    """
    if verbose: print(f"  Stage 1: clean pre-training ({N_EPOCHS_CLEAN} epochs) ...")
    net, p0_val = train_vanilla(
        S, V, VP, p0_init, n_epochs=N_EPOCHS_CLEAN, verbose=False
    )

    if verbose: print(f"  Stage 2: adversarial fine-tuning ({N_EPOCHS_ADV} epochs) ...")
    net = net.to(device)
    p0  = nn.Parameter(torch.tensor(p0_val, dtype=torch.float32, device=device))
    opt = torch.optim.Adam(
        [{'params': net.parameters()}, {'params': [p0]}], lr=LR_ADV
    )
    S_d, V_d, VP_d = S.to(device), V.to(device), VP.to(device)
    M = S_d.shape[0]

    for epoch in range(N_EPOCHS_ADV):
        net.train()
        idx = torch.randperm(M, device=device)[:min(BATCH_SIZE, M)]
        S_b, V_b, VP_b = S_d[idx], V_d[idx], VP_d[idx]

        # Clean loss
        h_c    = net(make_input(S_b, V_b))
        loss_c = loss_fn(h_c, S_b, VP_b, p0)

        # Inner maximisation: joint SV-attack (Corollary 4.2 of He et al.)
        p0_det = p0.detach()
        S_att, VP_att, _, _ = attacker.wp_attack(
            net, S_b, V_b, VP_b, loss_fn, p0_det,
            ATK_DELTA, ATK_RATIO, ATK_N_TRAIN,
        )

        # Outer minimisation: adversarial paths (V stays as true latent state)
        net.train()
        h_a    = net(make_input(S_att, V_b))
        loss_a = loss_fn(h_a, S_att, VP_att, p0)

        total = ALPHA_BAL * loss_c + loss_a
        opt.zero_grad(); total.backward(); opt.step()

        if verbose and (epoch % max(N_EPOCHS_ADV // 5, 1) == 0 or epoch == N_EPOCHS_ADV - 1):
            print(f"  adv epoch {epoch:3d}  total={total.item():.4f}  "
                  f"clean={loss_c.item():.4f}  adv={loss_a.item():.4f}")

    return net.cpu(), float(p0.item())


print("Training functions defined.")

## Experiment 1 — Bates Baseline: Vanilla vs W-DRO

In [ ]:
# ── Experiment 1: Train models on Bates paths ──────────────────────────────────
print("[Exp 1] Training Vanilla Bates ...")
t0 = time.perf_counter()
net_vanilla_bates, p0_vanilla_bates = train_vanilla(
    S_btr, V_btr, VP_btr, bates_p0_init,
    n_epochs=N_EPOCHS_CLEAN + N_EPOCHS_ADV,
)
print(f"  Done in {time.perf_counter()-t0:.0f}s")

torch.save(net_vanilla_bates.state_dict(), RESULTS_DIR / 'bates_vanilla_network.pt')
torch.save({
    'losses': [],
    'p0': p0_vanilla_bates,
    'alpha': ALPHA_CVAR,
    'vp_scale': bates_vp_scale,
}, RESULTS_DIR / 'bates_vanilla_log.pt')
print("  Saved → results/bates_vanilla_network.pt")

print("\n[Exp 1] Training W-DRO Bates ...")
t0 = time.perf_counter()
net_wdro_bates, p0_wdro_bates = train_wdro(
    S_btr, V_btr, VP_btr, bates_p0_init,
)
print(f"  Done in {time.perf_counter()-t0:.0f}s")

torch.save(net_wdro_bates.state_dict(), RESULTS_DIR / 'bates_wdro_network.pt')
torch.save({
    'losses': [],
    'p0': p0_wdro_bates,
    'alpha': ALPHA_CVAR,
    'vp_scale': bates_vp_scale,
    'atk_delta': ATK_DELTA,
}, RESULTS_DIR / 'bates_wdro_log.pt')
print("  Saved → results/bates_wdro_network.pt")

In [ ]:
# ── Experiment 1: Evaluate on Bates test set ───────────────────────────────────
print("[Exp 1] Evaluating on Bates test set ...")

err_vanilla_bates = compute_hedging_errors(net_vanilla_bates, S_bte, V_bte, VP_bte)
err_wdro_bates    = compute_hedging_errors(net_wdro_bates,    S_bte, V_bte, VP_bte)

m_vb = eval_metrics(err_vanilla_bates)
m_wb = eval_metrics(err_wdro_bates)

print("\n  Bates test set results:")
df1 = pd.DataFrame({'Vanilla Bates': m_vb, 'W-DRO Bates': m_wb}).T
print(df1.to_string(float_format='{:.4f}'.format))

In [ ]:
# ── Experiment 1: Plot PnL distributions ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

bins = 80
ax.hist((-err_vanilla_bates).numpy(), bins=bins, alpha=0.55,
        color='steelblue', label='Vanilla Bates', density=True)
ax.hist((-err_wdro_bates).numpy(),    bins=bins, alpha=0.55,
        color='darkorange', label='W-DRO Bates', density=True)

# Mark CVaR95 and VaR99
for m, col, lbl in [
    (m_vb, 'steelblue',  'Vanilla'),
    (m_wb, 'darkorange', 'W-DRO'),
]:
    ax.axvline(-m['cvar95'], color=col, linestyle='--', linewidth=1.5,
               label=f'CVaR95 {lbl} = {m["cvar95"]:.2f}')
    ax.axvline(-m['var99'],  color=col, linestyle=':',  linewidth=1.2)

ax.set_xlabel('P&L = −Hedging Error', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Exp 1 — Bates Baseline: Vanilla vs W-DRO\n'
             '(dashed=CVaR95, dotted=VaR99)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bates_figure1_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bates_figure1_baseline.png")

## Experiment 2 — Cross-Model Robustness: Heston → Bates

In [ ]:
# ── Experiment 2: Load Heston vanilla, train Heston W-DRO ─────────────────────
# Vanilla Heston: load from existing results (trained with alpha=0.95 on 100k paths)
heston_log = torch.load(RESULTS_DIR / 'heston_ES095_log.pt', weights_only=False)
heston_vp_scale_saved = heston_log['vp_scale']
p0_vanilla_heston = heston_log['p0']
print(f"Loaded heston_ES095_log: vp_scale={heston_vp_scale_saved:.2f}  p0={p0_vanilla_heston:.4f}")

net_vanilla_heston = HestonHedgeNet(N=N_STEPS, width=20)
net_vanilla_heston.load_state_dict(
    torch.load(RESULTS_DIR / 'heston_ES095_network.pt', weights_only=True)
)
net_vanilla_heston.eval()
print("Loaded heston_ES095_network.pt (vanilla Heston)")

# Re-scale Heston test paths with saved vp_scale for vanilla Heston model
# (the loaded model expects its own training scale)
VP_hte_for_hvanilla = VP_hte_raw * heston_vp_scale_saved
VP_bte_for_hvanilla = VP_bte_raw * heston_vp_scale_saved   # cross-model eval

print("\n[Exp 2] Training W-DRO Heston (fresh, same architecture) ...")
t0 = time.perf_counter()
# Use BatesHedgeNet — same architecture as HestonHedgeNet
# Re-scale Heston training VP with heston_vp_scale (fresh computation)
net_wdro_heston, p0_wdro_heston = train_wdro(
    S_htr, V_htr, VP_htr, heston_p0_init,
)
print(f"  Done in {time.perf_counter()-t0:.0f}s")

torch.save(net_wdro_heston.state_dict(), RESULTS_DIR / 'heston_wdro_network.pt')
torch.save({
    'p0': p0_wdro_heston, 'alpha': ALPHA_CVAR,
    'vp_scale': heston_vp_scale, 'atk_delta': ATK_DELTA,
}, RESULTS_DIR / 'heston_wdro_log.pt')
print("  Saved → results/heston_wdro_network.pt")

# W-DRO Heston paths use the freshly computed heston_vp_scale
VP_hte_for_hwdro = VP_htr_raw[:N_TE] * heston_vp_scale   # use same scale as training
VP_hte_for_hwdro_test = VP_hte_raw * heston_vp_scale
VP_bte_for_hwdro = VP_bte_raw * heston_vp_scale

In [ ]:
# ── Experiment 2: Evaluate all 4 models — in-sample and cross-model ─────────
print("[Exp 2] Evaluating all four models ...\n")

# In-sample evaluations
err_vb_insample  = err_vanilla_bates   # already computed
err_wb_insample  = err_wdro_bates      # already computed
err_vh_insample  = compute_hedging_errors(
    net_vanilla_heston, S_hte, V_hte, VP_hte_for_hvanilla
)
err_wh_insample  = compute_hedging_errors(
    net_wdro_heston, S_hte, V_hte, VP_hte_for_hwdro_test
)

# Cross-model: Heston-trained models evaluated on Bates test paths
# Apply each model's own vp_scale to Bates VP for PnL consistency
err_vh_bates = compute_hedging_errors(
    net_vanilla_heston, S_bte, V_bte, VP_bte_for_hvanilla
)
err_wh_bates = compute_hedging_errors(
    net_wdro_heston, S_bte, V_bte, VP_bte_for_hwdro
)

# Build results table
rows = {
    'Vanilla-Bates (in-sample)':    eval_metrics(err_vb_insample),
    'W-DRO-Bates (in-sample)':      eval_metrics(err_wb_insample),
    'Vanilla-Heston (in-sample)':   eval_metrics(err_vh_insample),
    'W-DRO-Heston (in-sample)':     eval_metrics(err_wh_insample),
    'Vanilla-Heston → Bates':       eval_metrics(err_vh_bates),
    'W-DRO-Heston → Bates':         eval_metrics(err_wh_bates),
}
df2 = pd.DataFrame(rows).T
print(df2.to_string(float_format='{:.4f}'.format))

# Degradation metric
delta_vanilla = rows['Vanilla-Heston → Bates']['cvar95'] - rows['Vanilla-Heston (in-sample)']['cvar95']
delta_wdro    = rows['W-DRO-Heston → Bates']['cvar95']   - rows['W-DRO-Heston (in-sample)']['cvar95']
print(f"\nΔCVaR (Heston→Bates): Vanilla={delta_vanilla:+.4f}  W-DRO={delta_wdro:+.4f}")
print("(W-DRO should degrade less — smaller ΔCVAR indicates better cross-model robustness)")

In [ ]:
# ── Experiment 2: 4-panel PnL histogram ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=False)

panels = [
    (axes[0, 0], err_vh_insample,  'Vanilla-Heston (in-sample)',  'steelblue'),
    (axes[0, 1], err_wh_insample,  'W-DRO-Heston (in-sample)',    'darkorange'),
    (axes[1, 0], err_vh_bates,     'Vanilla-Heston → Bates',      'steelblue'),
    (axes[1, 1], err_wh_bates,     'W-DRO-Heston → Bates',        'darkorange'),
]

for ax, err, title, col in panels:
    m = eval_metrics(err)
    ax.hist((-err).numpy(), bins=60, alpha=0.65, color=col, density=True)
    ax.axvline(-m['cvar95'], color='red',   linestyle='--', linewidth=1.8,
               label=f'CVaR95={m["cvar95"]:.2f}')
    ax.axvline(-m['var99'],  color='black', linestyle=':',  linewidth=1.3,
               label=f'VaR99={m["var99"]:.2f}')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('P&L', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25, linestyle='--')

fig.suptitle(
    'Exp 2 — Cross-Model Robustness: Heston → Bates\n'
    'Left tail worsening under Bates is expected (jump risk is unhedgeable)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bates_figure2_crossmodel.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bates_figure2_crossmodel.png")

## Experiment 3 — Jump Intensity Stress Test (λ)

In [ ]:
# ── Experiment 3: Sweep λ, evaluate all 4 models ──────────────────────────────
print("[Exp 3] Jump intensity stress test ...\n")

stress_results = {lam: {} for lam in LAMBDA_GRID}

for lam in LAMBDA_GRID:
    t0 = time.perf_counter()
    stress_params = BatesParams(
        S0=BATES_S0, v0=BATES_V0, kappa=BATES_KAPPA, theta=BATES_THETA,
        xi=BATES_XI, rho=BATES_RHO, mu_J=BATES_MU_J, sigma_J=BATES_SIGMA_J,
        lam=lam, T=N_STEPS/365, N=N_STEPS, M=N_STRESS_TE,
    )
    S_s, V_s, VP_s_raw = BatesSimulator(stress_params).simulate(seed=300 + int(lam * 100))

    # Apply each model's own vp_scale consistently
    VP_s_bates  = VP_s_raw * bates_vp_scale
    VP_s_heston = VP_s_raw * heston_vp_scale_saved   # for vanilla Heston
    VP_s_hwdro  = VP_s_raw * heston_vp_scale          # for W-DRO Heston

    err_vb  = compute_hedging_errors(net_vanilla_bates,  S_s, V_s, VP_s_bates)
    err_wb  = compute_hedging_errors(net_wdro_bates,     S_s, V_s, VP_s_bates)
    err_vh  = compute_hedging_errors(net_vanilla_heston, S_s, V_s, VP_s_heston)
    err_wh  = compute_hedging_errors(net_wdro_heston,    S_s, V_s, VP_s_hwdro)

    stress_results[lam] = {
        'Vanilla-Bates':  {'cvar95': empirical_ES(err_vb), 'var99': float(np.quantile(err_vb.numpy(), 0.99))},
        'W-DRO-Bates':    {'cvar95': empirical_ES(err_wb), 'var99': float(np.quantile(err_wb.numpy(), 0.99))},
        'Vanilla-Heston': {'cvar95': empirical_ES(err_vh), 'var99': float(np.quantile(err_vh.numpy(), 0.99))},
        'W-DRO-Heston':   {'cvar95': empirical_ES(err_wh), 'var99': float(np.quantile(err_wh.numpy(), 0.99))},
    }
    print(f"  λ={lam:.2f}  VB={stress_results[lam]['Vanilla-Bates']['cvar95']:.3f}  "
          f"WB={stress_results[lam]['W-DRO-Bates']['cvar95']:.3f}  "
          f"VH={stress_results[lam]['Vanilla-Heston']['cvar95']:.3f}  "
          f"WH={stress_results[lam]['W-DRO-Heston']['cvar95']:.3f}  "
          f"({time.perf_counter()-t0:.1f}s)")

In [ ]:
# ── Experiment 3: Robustness curves ───────────────────────────────────────────
MODELS = ['Vanilla-Bates', 'W-DRO-Bates', 'Vanilla-Heston', 'W-DRO-Heston']
COLORS = ['steelblue', 'darkorange', 'seagreen', 'crimson']
STYLES = ['-', '--', '-.', ':']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model, col, ls in zip(MODELS, COLORS, STYLES):
    cvar_vals = [stress_results[lam][model]['cvar95'] for lam in LAMBDA_GRID]
    var99_vals = [stress_results[lam][model]['var99']  for lam in LAMBDA_GRID]
    ax1.plot(LAMBDA_GRID, cvar_vals,  color=col, linestyle=ls, marker='o',
             linewidth=2, markersize=5, label=model)
    ax2.plot(LAMBDA_GRID, var99_vals, color=col, linestyle=ls, marker='s',
             linewidth=2, markersize=5, label=model)

ax1.axvline(BATES_LAM, color='grey', linestyle=':', alpha=0.6, label=f'Calibrated λ={BATES_LAM}')
ax2.axvline(BATES_LAM, color='grey', linestyle=':', alpha=0.6)

ax1.set_xlabel('Jump Intensity λ', fontsize=12)
ax1.set_ylabel('CVaR₀.₉₅', fontsize=12)
ax1.set_title('(a) CVaR₀.₉₅ vs Jump Intensity', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3, linestyle='--')

ax2.set_xlabel('Jump Intensity λ', fontsize=12)
ax2.set_ylabel('VaR₀.₉₉', fontsize=12)
ax2.set_title('(b) VaR₀.₉₉ vs Jump Intensity', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, linestyle='--')

fig.suptitle('Exp 3 — Jump Intensity Stress Test\n'
             'Shallower slope = better robustness to increasing jump risk', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bates_figure3_lambda_stress.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bates_figure3_lambda_stress.png")

# Relative robustness ratios
print("\nRelative Robustness Ratio RR(λ) = CVaR_WDRO / CVaR_Vanilla:")
for lam in LAMBDA_GRID:
    rr_b = stress_results[lam]['W-DRO-Bates']['cvar95']   / (stress_results[lam]['Vanilla-Bates']['cvar95']   + 1e-9)
    rr_h = stress_results[lam]['W-DRO-Heston']['cvar95']  / (stress_results[lam]['Vanilla-Heston']['cvar95']  + 1e-9)
    print(f"  λ={lam:.2f}  RR_Bates={rr_b:.3f}  RR_Heston={rr_h:.3f}")

## Experiment 4 — Adversarial Attacks on Bates

In [ ]:
# ── Experiment 4: S-WPGD and SV-WPGD attacks ──────────────────────────────────
print("[Exp 4] Adversarial attacks on Bates test paths ...\n")

# Use a fixed subset for faster attack evaluation
N_ATK = min(N_TE, 5_000 if QUICK_RUN else 10_000)
S_atk  = S_bte[:N_ATK].to(device)
V_atk  = V_bte[:N_ATK].to(device)
VP_atk = VP_bte[:N_ATK].to(device)

net_vb_dev = net_vanilla_bates.to(device)
net_wb_dev = net_wdro_bates.to(device)

p0_vb_t = torch.tensor(p0_vanilla_bates, dtype=torch.float32, device=device)
p0_wb_t = torch.tensor(p0_wdro_bates,    dtype=torch.float32, device=device)

attack_rows = []

for delta in DELTA_GRID:
    row = {'delta': delta}
    print(f"  δ={delta:.2f}", end='', flush=True)

    if delta == 0.0:
        # Baseline (no attack)
        err_vb_0 = compute_hedging_errors(net_vb_dev, S_atk, V_atk, VP_atk)
        err_wb_0 = compute_hedging_errors(net_wb_dev, S_atk, V_atk, VP_atk)
        for col in ['S-WPGD Vanilla', 'S-WPGD W-DRO', 'SV-WPGD Vanilla', 'SV-WPGD W-DRO']:
            row[col] = empirical_ES(err_vb_0 if 'Vanilla' in col else err_wb_0)
    else:
        # S-only attack
        S_att_vb, VP_att_vb, _, _ = attacker.wp_attack(
            net_vb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_vb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=True,
        )
        S_att_wb, VP_att_wb, _, _ = attacker.wp_attack(
            net_wb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_wb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=True,
        )
        row['S-WPGD Vanilla'] = empirical_ES(
            compute_hedging_errors(net_vb_dev, S_att_vb, V_atk, VP_atk)
        )
        row['S-WPGD W-DRO'] = empirical_ES(
            compute_hedging_errors(net_wb_dev, S_att_wb, V_atk, VP_atk)
        )
        print(f"  S: vb={row['S-WPGD Vanilla']:.3f} wb={row['S-WPGD W-DRO']:.3f}", end='', flush=True)

        # SV joint attack
        S_att_sv_vb, VP_att_sv_vb, _, _ = attacker.wp_attack(
            net_vb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_vb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=False,
        )
        S_att_sv_wb, VP_att_sv_wb, _, _ = attacker.wp_attack(
            net_wb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_wb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=False,
        )
        row['SV-WPGD Vanilla'] = empirical_ES(
            compute_hedging_errors(net_vb_dev, S_att_sv_vb, V_atk, VP_att_sv_vb)
        )
        row['SV-WPGD W-DRO'] = empirical_ES(
            compute_hedging_errors(net_wb_dev, S_att_sv_wb, V_atk, VP_att_sv_wb)
        )
        print(f"  SV: vb={row['SV-WPGD Vanilla']:.3f} wb={row['SV-WPGD W-DRO']:.3f}")

    attack_rows.append(row)

df_attack = pd.DataFrame(attack_rows).set_index('delta')
df_attack.to_csv(RESULTS_DIR / 'bates_table1_attacks.csv')
print("\nAttack table saved → results/bates_table1_attacks.csv")
print(df_attack.to_string(float_format='{:.4f}'.format))

In [ ]:
# ── Experiment 4: Frobenius distance table (mirrors He et al. Table 2) ─────────
print("[Exp 4] Covariance Frobenius distances ...\n")

def covariance_frobenius(
    orig: torch.Tensor,     # (batch, T, F) or (batch, T)
    attacked: torch.Tensor,
) -> float:
    """||Sigma_orig - Sigma_att||_F where Sigma is sample covariance over paths."""
    o = orig.reshape(orig.shape[0], -1).numpy()
    a = attacked.reshape(attacked.shape[0], -1).numpy()
    Sigma_o = np.cov(o, rowvar=False)
    Sigma_a = np.cov(a, rowvar=False)
    return float(np.linalg.norm(Sigma_o - Sigma_a, 'fro'))


# Use vanilla model attacks as representative (same perturbation structure)
frob_rows = []
for delta in DELTA_GRID:
    row = {'delta': delta}
    if delta == 0.0:
        row['S-WPGD Frob']  = 0.0
        row['SV-WPGD Frob'] = 0.0
    else:
        S_att_s, _, _, _   = attacker.wp_attack(
            net_vb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_vb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=True,
        )
        S_att_sv, VP_att_sv, _, _ = attacker.wp_attack(
            net_vb_dev, S_atk, V_atk, VP_atk, loss_fn, p0_vb_t,
            delta, ATK_RATIO, ATK_N_EVAL, s_only=False,
        )
        row['S-WPGD Frob']  = covariance_frobenius(S_atk.cpu(), S_att_s.cpu())
        # For SV, use joint [S, VP] as the feature matrix
        orig_sv = torch.stack([S_atk.cpu(), VP_atk.cpu()], dim=-1)
        att_sv  = torch.stack([S_att_sv.cpu(), VP_att_sv.cpu()], dim=-1)
        row['SV-WPGD Frob'] = covariance_frobenius(orig_sv, att_sv)
    frob_rows.append(row)

df_frob = pd.DataFrame(frob_rows).set_index('delta')
print(df_frob.to_string(float_format='{:.4f}'.format))

In [ ]:
# ── Experiment 4: CVaR vs δ curves ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, col_v, col_w, title in [
    (ax1, 'S-WPGD Vanilla', 'S-WPGD W-DRO',  '(a) S-WPGD Attack'),
    (ax2, 'SV-WPGD Vanilla', 'SV-WPGD W-DRO', '(b) SV-WPGD Attack'),
]:
    ax.plot(df_attack.index, df_attack[col_v], 'o-', color='steelblue',
            linewidth=2, markersize=5, label='Vanilla Bates')
    ax.plot(df_attack.index, df_attack[col_w], 's--', color='darkorange',
            linewidth=2, markersize=5, label='W-DRO Bates')
    ax.set_xlabel('Wasserstein radius δ', fontsize=12)
    ax.set_ylabel('CVaR₀.₉₅ under attack', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xscale('symlog', linthresh=0.01)

fig.suptitle('Exp 4 — Adversarial Attacks on Bates\n'
             'Gap between curves is the main robustness claim', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bates_figure4_attacks.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bates_figure4_attacks.png")

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("BATES EXPERIMENTS SUMMARY")
print("=" * 60)
print(f"\nExp 1 — Bates Baseline (N_TR={N_TR:,})")
print(f"  Vanilla CVaR95 = {m_vb['cvar95']:.4f}")
print(f"  W-DRO   CVaR95 = {m_wb['cvar95']:.4f}")

print(f"\nExp 2 — Cross-Model Robustness")
print(f"  ΔCVAR Vanilla (Heston→Bates) = {delta_vanilla:+.4f}")
print(f"  ΔCVAR W-DRO   (Heston→Bates) = {delta_wdro:+.4f}")
if abs(delta_vanilla) > 1e-6:
    print(f"  W-DRO degrades {100*(1 - abs(delta_wdro)/(abs(delta_vanilla)+1e-9)):.1f}% less")

print(f"\nExp 3 — Lambda Stress Test")
for lam in LAMBDA_GRID:
    rr = (stress_results[lam]['W-DRO-Bates']['cvar95'] /
          (stress_results[lam]['Vanilla-Bates']['cvar95'] + 1e-9))
    print(f"  λ={lam:.2f}  RR={rr:.3f}")

print(f"\nExp 4 — Adversarial Attacks at δ=0.1")
if 0.1 in df_attack.index:
    row = df_attack.loc[0.1]
    print(f"  S-WPGD:  Vanilla={row['S-WPGD Vanilla']:.4f}  W-DRO={row['S-WPGD W-DRO']:.4f}")
    print(f"  SV-WPGD: Vanilla={row['SV-WPGD Vanilla']:.4f}  W-DRO={row['SV-WPGD W-DRO']:.4f}")

print("\nSaved files:")
for f in sorted(RESULTS_DIR.glob('bates_*')):
    print(f"  {f.name}")